# Lab 4: Prove it

**AI in Space Applications II · after Lesson 24 · 30 minutes · open**

**Goal.** Turn "it works" into evidence a review board can check: one test and one pass criterion for each of R1 to R6, the rule of three behind every failure-rate claim, a clean rerun, and a one-page model card.

This lab is **open**: it gives you the structure and the helper functions, and the tests are yours to write. The reference solution's tests are a guide, not the only answer.

**You need** `vega1_train.npz`, `vega1_test.npz`, your model checkpoint (`compact_model.pt` from Lab 2, or `lab1_model.pt`), and `lab1_results.json`, `lab2_results.json`, `lab3_results.json`. If you ran the earlier labs in Colab with Drive connected, all of these are in your Drive folder; the data files download by themselves.

**In Colab.** Opened this from the course's *Open in Colab* link? First choose **File → Save a copy in Drive**, or your edits are lost when you close the tab. When the setup cell asks to connect Google Drive, allow it: the notebook keeps your models and results in the Drive folder `AI_in_Space_II_labs`, where later labs and sessions find them. The data files download by themselves.

In [ ]:
import json
import os
import platform
import random
import shutil
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as Fn

FAST = os.environ.get("VEGA1_FAST") == "1"        # set only by the course's own tests: a short run that checks the code
SEED = 0

KEEP_DIR = None                                   # your folder in Google Drive, when this runs in Colab
try:
    from google.colab import drive                # Colab only: Colab forgets files when a session ends, Drive does not
    drive.mount("/content/drive")
    KEEP_DIR = "/content/drive/MyDrive/AI_in_Space_II_labs"
    os.makedirs(KEEP_DIR, exist_ok=True)
    for name in os.listdir(KEEP_DIR):             # bring back what you saved in earlier labs and sessions
        if not os.path.exists(name) and os.path.isfile(os.path.join(KEEP_DIR, name)):
            shutil.copy2(os.path.join(KEEP_DIR, name), name)
    print("Your Drive folder:", KEEP_DIR, "| files found there:", ", ".join(sorted(os.listdir(KEEP_DIR))) or "none yet")
except ImportError:
    pass                                          # not in Colab: files stay beside the notebook
except Exception as e:
    print("Google Drive is not connected, so files last only for this session:", e)


def keep(*names):
    """Copy these files to your Drive folder, so the next session and the next lab find them."""
    if KEEP_DIR:
        for name in names:
            if os.path.exists(name):
                shutil.copy2(name, os.path.join(KEEP_DIR, name))
        print("Kept in your Drive folder:", ", ".join(names))


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Python", platform.python_version(), "| PyTorch", torch.__version__, "| NumPy", np.__version__, "| device:", DEVICE)

In [ ]:
DATA_DIR = os.environ.get("VEGA1_DATA_DIR", ".")
DATA_URL = "https://github.com/Astronify-SpaceTech/ai-in-space-ii-labs/releases/download/data-v1/"   # the course's data files


def get_file(name):
    """Find a course data file: beside the notebook, else download it from the course's link, else upload it in Colab."""
    path = os.path.join(DATA_DIR, name)
    if os.path.exists(path):
        return path
    if DATA_URL:
        import urllib.request
        try:
            print(f"downloading {name} ...")
            urllib.request.urlretrieve(DATA_URL + name, path + ".part")
            os.replace(path + ".part", path)
            return path
        except Exception as e:
            print(f"download failed ({e}); choose the file you downloaded from the lesson page instead")
    try:
        from google.colab import files       # Colab only: choose the file you downloaded from the lesson page
        files.upload()
        return name
    except ImportError:
        raise FileNotFoundError(f"Put {name} beside this notebook (it is attached to the lab on the course page).")


d = np.load(get_file("vega1_train.npz"))
X = d["X"].astype(np.float32) * float(d["scale"]) + float(d["offset"])     # stored value x scale + offset = reflectance
y = d["y"].astype(np.int64)                      # 0 clear, 1 thick cloud, 2 thin cloud, 3 cloud shadow
region, patch = d["region"], d["patch"]
CLASSES = ["clear", "thick cloud", "thin cloud", "cloud shadow"]

clear_share = (y == 0).mean(axis=(1, 2))         # share of each chip that is usable ground
usable = clear_share >= 0.5                      # R2 counts chips with at least half their area usable
print(f"{len(X)} chips, {X.shape[1]} bands, {X.shape[2]} x {X.shape[3]} pixels, {len(np.unique(region))} regions, "
      f"{len(np.unique(patch))} patches")

In [ ]:
class Block(nn.Sequential):
    def __init__(self, cin, cout):
        super().__init__(nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                         nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))


class SmallUNet(nn.Module):
    """Input: (batch, 7, H, W) reflectance. Output: (batch, 4, H, W) class scores. H and W must divide by 4."""

    def __init__(self, bands=7, classes=4, widths=(16, 32, 64)):
        super().__init__()
        a, b, c = widths
        self.register_buffer("mean", torch.zeros(1, bands, 1, 1))
        self.register_buffer("std", torch.ones(1, bands, 1, 1))
        self.enc1, self.enc2, self.mid = Block(bands, a), Block(a, b), Block(b, c)
        self.dec2, self.dec1 = Block(c + b, b), Block(b + a, a)
        self.head = nn.Conv2d(a, classes, 1)

    def features(self, x):
        x = (x - self.mean) / self.std
        e1 = self.enc1(x)
        e2 = self.enc2(Fn.max_pool2d(e1, 2))
        m = self.mid(Fn.max_pool2d(e2, 2))
        return e1, e2, m

    def forward(self, x):
        e1, e2, m = self.features(x)
        d2 = self.dec2(torch.cat([Fn.interpolate(m, scale_factor=2.0, mode="nearest"), e2], dim=1))
        d1 = self.dec1(torch.cat([Fn.interpolate(d2, scale_factor=2.0, mode="nearest"), e1], dim=1))
        return self.head(d1)


def train_model(X_train, y_train, make_model=SmallUNet, epochs=None, seed=SEED):
    """Train a model on the given chips and return it. About a minute on a GPU, a few minutes on a CPU."""
    set_seed(seed)
    epochs = epochs or (1 if FAST else 6)
    model = make_model().to(DEVICE)
    model.mean.copy_(torch.tensor(X_train.mean(axis=(0, 2, 3))).view(1, -1, 1, 1))      # normalise with the training
    model.std.copy_(torch.tensor(X_train.std(axis=(0, 2, 3)) + 1e-6).view(1, -1, 1, 1))  # chips' own statistics
    opt = torch.optim.Adam(model.parameters(), lr=2e-3)
    xt, yt = torch.from_numpy(X_train), torch.from_numpy(y_train)
    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(xt))
        total = 0.0
        for k in range(0, len(order), 16):
            idx = order[k:k + 16]
            xb, yb = xt[idx].to(DEVICE), yt[idx].to(DEVICE)
            if random.random() < 0.5:                     # a mirror flip: the same chip, seen another way
                xb, yb = xb.flip(-1), yb.flip(-1)
            logits = model(xb)
            step = yb.shape[-1] // logits.shape[-1]       # a model with a coarser output is scored on coarser labels
            loss = Fn.cross_entropy(logits, yb[:, ::step, ::step])
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += float(loss) * len(idx)
        print(f"  epoch {epoch + 1}/{epochs}: loss {total / len(order):.3f} ({time.time() - t0:.0f} s)")
    return model.eval()


@torch.no_grad()
def predict(model, X_any, batch=32):
    """Class probabilities for every pixel, at the chip's own size: (chips, 4, H, W)."""
    out = []
    for k in range(0, len(X_any), batch):
        xb = torch.from_numpy(X_any[k:k + batch]).to(DEVICE)
        p = torch.softmax(model(xb), dim=1)
        if p.shape[-1] != xb.shape[-1]:
            p = Fn.interpolate(p, size=xb.shape[-2:], mode="nearest")
        out.append(p.cpu().numpy())
    return np.concatenate(out)

In [ ]:
def scores(model, X_test, y_test):
    """Cloud IoU (thick and thin cloud together), usable-area error in points, and pixel accuracy."""
    prob = predict(model, X_test)
    pred = prob.argmax(axis=1)
    pred_cloud, true_cloud = np.isin(pred, (1, 2)), np.isin(y_test, (1, 2))
    iou = (pred_cloud & true_cloud).sum() / max((pred_cloud | true_cloud).sum(), 1)
    area_error = np.abs(prob[:, 0].mean(axis=(1, 2)) - (y_test == 0).mean(axis=(1, 2))).mean() * 100
    return {"cloud_iou": float(iou), "usable_area_error_points": float(area_error),
            "pixel_accuracy": float((pred == y_test).mean())}


def region_split(region, share=0.25, seed=SEED):
    """Hold out whole regions: (train indices, test indices, the held-out regions)."""
    regions = np.random.default_rng(seed).permutation(np.unique(region))
    test_regions = regions[:max(1, int(len(regions) * share))]
    is_test = np.isin(region, test_regions)
    return np.where(~is_test)[0], np.where(is_test)[0], test_regions

## 1. The helpers

- `rule_of_three_bound(n)`: the exact upper bound on a failure rate after n clean, independent tests, at 95% confidence (Lesson 24).
- `tests_needed(p)`: how many clean tests a promise of "at most p" needs.
- `record(...)`: adds a row to the test table.

In [ ]:
import hashlib


def rule_of_three_bound(n, alpha=0.05):
    """No failures in n independent tests: the failure rate is below this, with 1 - alpha confidence."""
    return 1.0 - alpha ** (1.0 / n)


def tests_needed(p_max, alpha=0.05):
    return int(np.ceil(np.log(alpha) / np.log1p(-p_max) - 1e-12))


TABLE = []


def record(req, test, data, criterion, result, passed, evidence):
    TABLE.append({"requirement": req, "test": test, "data": data, "criterion": criterion, "result": result,
                  "pass": bool(passed), "evidence": evidence})
    print(f"{req}: {'PASS' if passed else 'FAIL'} | {result}")


def load_json(name):
    with open(name, encoding="utf-8") as f:
        return json.load(f)


lab1, lab2, lab3 = load_json("lab1_results.json"), load_json("lab2_results.json"), load_json("lab3_results.json")
ck_path = "compact_model.pt" if os.path.exists("compact_model.pt") and lab2.get("choice", "").startswith("compact") else "lab1_model.pt"
print("checkpoint used:", ck_path)

## 2. Load the model you carried forward

The compact model and the Lab 1 U-Net are both defined here, so either checkpoint loads.

In [ ]:
def sep(cin, cout):
    return nn.Sequential(nn.Conv2d(cin, cin, 3, padding=1, groups=cin, bias=False), nn.Conv2d(cin, cout, 1, bias=False),
                         nn.BatchNorm2d(cout), nn.ReLU(inplace=True))


class CompactNet(nn.Module):
    def __init__(self, bands=7, classes=4, widths=(16, 32, 64)):
        super().__init__()
        a, b, c = widths
        self.register_buffer("mean", torch.zeros(1, bands, 1, 1))
        self.register_buffer("std", torch.ones(1, bands, 1, 1))
        self.enc1 = nn.Sequential(nn.Conv2d(bands, a, 3, padding=1, bias=False), nn.BatchNorm2d(a), nn.ReLU(inplace=True))
        self.enc2 = sep(a, b)
        self.mid = nn.Sequential(sep(b, c), sep(c, c))
        self.dec = sep(c + b, b)
        self.head = nn.Conv2d(b, classes, 1)

    def forward(self, x):
        x = (x - self.mean) / self.std
        e1 = self.enc1(x)
        e2 = self.enc2(Fn.max_pool2d(e1, 2))
        m = self.mid(Fn.max_pool2d(e2, 2))
        return self.head(self.dec(torch.cat([Fn.interpolate(m, scale_factor=2.0, mode="nearest"), e2], dim=1)))


ck = torch.load(ck_path, map_location="cpu")
model = (CompactNet if ck_path.startswith("compact") else SmallUNet)(widths=tuple(ck["widths"]))
model.load_state_dict(ck["state"])
model.eval()
test_regions = np.array(ck["test_regions"])
is_test = np.isin(region, test_regions)
train_idx, test_idx = np.where(~is_test)[0], np.where(is_test)[0]

## 3. Core task: one test per requirement

For each requirement, write a test whose pass criterion is a number, and record it. Use the evidence your labs saved, and measure again where you can. **R6 must include a test on data unlike the training data:** the independent test set was labelled by another team, with another class list (thick and thin cloud merged into "cloud"), and its pixels are a different size.

In [ ]:
# R1: honest triage, on the held-out regions, against the pass mark committed in Lab 1
your R1 test: measure, compare with the pass mark you committed, then record(...)

In [ ]:
# R2: cost-based threshold: at most 2% of mostly usable chips discarded
your R2 test

In [ ]:
# R3: fits the flight computer (memory and the 99th-percentile scene time, from Lab 2's benchmark)
your R3 test

In [ ]:
# R4: survives a bit flip (Lab 2's test with the chosen mitigation)
your R4 test

In [ ]:
# R5: knows when it doesn't know (Lab 3's commitments and results, kind by kind)
your R5 test

**R6, the test on data unlike the training data.** Write down the largest drop you will accept **before** you run the cell. The independent test set has three classes, so compare cloud IoU, which means the same in both.

In [ ]:
ACCEPTABLE_DROP = ____                    # the largest fall in cloud IoU you accept on unfamiliar data: commit it first
t = np.load(get_file("vega1_test.npz"))
XT = t["X"].astype(np.float32) * float(t["scale"]) + float(t["offset"])
yt3 = t["y"].astype(np.int64)
not_snow = ~t["snow"].astype(bool)
prob = predict(model, XT[not_snow])
pc, tc = np.isin(prob.argmax(1), (1, 2)), yt3[not_snow] == 1
iou_other = float((pc & tc).sum() / max((pc | tc).sum(), 1))
drop = s_r1["cloud_iou"] - iou_other
record("R6a", "cloud IoU on another team's labels (no snow)", f"{not_snow.sum()} chips, Cloud Mask Catalogue",
       f"a fall of at most {ACCEPTABLE_DROP}", f"{s_r1['cloud_iou']:.2f} -> {iou_other:.2f} ({drop:+.2f})", drop <= ACCEPTABLE_DROP,
       "lab4 section 3")

## 4. The rule of three, on your own claims

Every failure-rate claim needs its count of **independent** tests. Chips cut from one picture are not independent: count pictures (subscenes), not chips.

**Your turn.** For R5's snow, how many snow subscenes did your monitor flag with no miss, and what miss rate can you claim? How many would a promise of "fewer than one snowy scene in ten" need?

In [ ]:
snow_scenes = t["subscene"][t["snow"].astype(bool)]
n_clean = ____                  # independent snow subscenes the monitor flagged with no miss
need = tests_needed(0.10)
print(f"{len(snow_scenes)} snow chips from {len(np.unique(snow_scenes))} subscenes; clean subscenes: {n_clean}")
if n_clean:
    claim = rule_of_three_bound(n_clean)
    result = f"miss rate below {claim:.0%} at 95% confidence (a promise of 1 in 10 needs {need})"
else:
    result = f"no clean snow subscene: no failure-rate claim can be made (a promise of 1 in 10 needs {need})"
print(result)
record("R6b", "rule of three on the snow claim", f"{n_clean} independent snow subscenes", "a stated bound backed by its count",
       result, n_clean > 0, "lab4 section 4")

## 5. Your turn: a clean rerun, and the model card

Restart the runtime (Runtime → Restart session), run every cell from the top, and check that the fingerprint below matches the one from your first run. Seeds and versions are recorded with it. Then fill the model card and commit it.

In [ ]:
fingerprint = hashlib.sha256(json.dumps(TABLE, sort_keys=True, default=str).encode()).hexdigest()[:12]
versions = {"python": platform.python_version(), "torch": torch.__version__, "numpy": np.__version__}
print("test-table fingerprint:", fingerprint, "| seed", SEED, "|", versions)

card = f"""# Model card: VEGA-1 cloud triage

| | |
|---|---|
| Model | {ck_path}, seed {SEED} |
| What it does | labels every pixel clear, thick cloud, thin cloud or shadow; the clear share decides send or discard |
| Intended use | onboard triage of VEGA-1 tiles: send, discard, or flag for a person |
| Not for | other sensors; decisions with no person in the loop for flagged tiles |
| Training data | VEGA-1 training set (CloudSEN12+ via Copernicus-Bench Cloud-S2, CC0), bands B02 to B08 |
| Evaluation | held-out regions; another team's labels (Cloud Mask Catalogue, CC BY 4.0); stress tiles |

## Results

""" + "\n".join(f"- {r['requirement']}: {r['result']} ({'pass' if r['pass'] else 'fail'})" for r in TABLE) + f"""

## Limits

- No shortwave infrared: snow and cloud are hard to tell apart (the sensor gap).
- Trained on 20 m pixels; the flight graph averages VEGA-1's tiles 4 x 4 to match.

## Reproduce

Run lab1 to lab4 in order; seed {SEED}; {versions}; test-table fingerprint {fingerprint}.
"""
with open("MODEL_CARD.md", "w", encoding="utf-8") as f:
    f.write(card)
with open("lab4_results.json", "w", encoding="utf-8") as f:
    json.dump({"table": TABLE, "fingerprint": fingerprint, "versions": versions}, f, indent=1)
print(f"{sum(r['pass'] for r in TABLE)} of {len(TABLE)} tests pass; saved MODEL_CARD.md and lab4_results.json")
keep("MODEL_CARD.md", "lab4_results.json")

## Check, then break

**Check:** the bound for 30 clean tests is close to 3 ÷ 30. **Break:** count the 16 chips of one snow subscene as 16 tests, and see how far the claim inflates.

In [ ]:
print(f"30 clean tests: exact {rule_of_three_bound(30):.3f}, rule of three {3 / 30:.3f}")
print(f"one subscene counted as 16 tests: 'below {rule_of_three_bound(16):.0%}'; counted honestly as 1 test: 'below {rule_of_three_bound(1):.0%}'")

## Stretch (optional, not graded)

- **A Kalman filter in NumPy** (Lesson 21): fuse a drifting gyro with a noisy star tracker; check that the normalised innovations stay inside their band, then set Q to zero and watch the filter grow confident and wrong.
- **A shielded lander** (Lesson 23): train a small agent on Gymnasium's LunarLander and put a shield in front of it that blocks a fast descent near the ground. Count how often the shield steps in.